In [1]:
%pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


# Для всех датасетов в начале файла разделить их на 
- A с фактором соседства с В
- B с фактором соседства с А
- A * B
### И обрабатывать по отдельнтости каждый своей моделью

Например, у Fligts и Operational есть фактор flight
нужно сделать 3 датафрейма
1. Flights с доп бинарным признаком "Есть ли данные обо мне в Operational"
2. Operational с доп бинарным признаком "Есть ли данные обо мне в Flights"
3. Их пересечение

Потом будут рассматриваться как отдельные датасеты

Также и для Boarding и для всех

По итогу выйдут 7 датасетов исходных с соседствами
и всевозможные их пересечения (попарные, по тройкам и так далее)

выделять пересечения в отдельный датасет можно лишь если > 1000

In [2]:
import pandas as pd

path = "../../resources/processed/temp/"

BoardingDF = pd.read_csv(f"{path}BoardingData_parsed.csv", sep="|", engine="pyarrow")
FlightsDF = pd.read_csv(f"{path}json_parsed.csv", sep="|", engine="pyarrow")
OperationalDF = pd.read_csv(f"{path}yaml_parsed.csv", sep="|", engine="pyarrow")
TimetableDF = pd.read_csv(f"{path}Skyteam_Timetable.csv", sep="|", engine="pyarrow")
SirenaDF = pd.read_csv(f"{path}sirena_parsed.csv", sep="|", engine="pyarrow")
BoardingPassDF = pd.read_csv(f"{path}cleaned_boarding_pass.csv", sep="|", engine="pyarrow")
LoyalityDF = pd.read_csv(f"{path}xml_parsed.csv", sep="|", engine="pyarrow")

In [3]:
display(BoardingDF)

,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,PassengerBirthDate,PassengerDocument,BookingCode,TicketNumber,Baggage,FlightDate,FlightTime,FlightNumber,CodeShare,Destination
0,SAVELII,VIKTOROVICH,RUSANOV,Male,03/10/1983,2879 096860,FRNINO,6625956945991971,Transit,2017-03-22,06:05:00,SU1369,Own,Moscow
1,LEV,MARKOVICH,ISAEV,Male,12/13/1975,1788 173211,Not presented,1643715499224676,Registered,2017-03-18,22:10:00,SU1180,Own,Volgograd
2,NIKOLAI,I.,NIKOLSKII,Male,12/26/1990,4396 926588,VWNYGF,6247422701565929,Transit,2017-03-18,22:10:00,SU1180,Own,Volgograd
3,ANATOLII,PETROVICH,SHILOV,Male,05/24/1997,2595 919752,WQFFUE,Not presented,Registered,2017-03-18,22:10:00,SU1180,Own,Volgograd
4,MIROSLAVA,VIACHESLAVOVNA,SEMENOVA,Female,01/31/1976,6775 516990,Not presented,Not presented,Registered,2017-03-12,11:45:00,SU6284,Own,Moscow
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155142,RAMIL,S.,DOROFEEV,Male,02/14/1994,3256 077151,SNRBXU,8653871661040452,Delayed,2017-05-18,06:30:00,SU1158,Own,Rostov
155143,RUSLAN,KONSTANTINOVICH,CHISTIAKOV,Male,02/14/1975,3497 458395,Not presented,0223584175158947,NaN,2017-05-18,06:30:00,SU1158,Own,Rostov
155144,SOFIA,VALEREVNA,KALMYKOVA,Female,12/17/1972,6618 352713,Not presented,Not presented,Registered,2017-10-02,07:30:00,SU5614,Own,Petropavlovsk
155145,VIACHESLAV,F.,KONONOV,Male,05/20/2000,4567 242088,Not presented,0975693658645383,Registered,2017-09-16,07:00:00,SU1640,Own,Omsk


In [4]:
BoardingDF["PassengerFirstName"] = BoardingDF['PassengerFirstName'].map(
    BoardingDF['PassengerFirstName'].value_counts(normalize=True)
)
BoardingDF["PassengerSecondName"] = BoardingDF['PassengerSecondName'].map(
    BoardingDF['PassengerSecondName'].value_counts(normalize=True)
)
BoardingDF["PassengerLastName"] = BoardingDF['PassengerLastName'].map(
    BoardingDF['PassengerLastName'].value_counts(normalize=True)
)

BoardingDF["PassengerSex"] = BoardingDF["PassengerSex"].map({'Female': 0, 'Male': 1})

BoardingDF['PassengerBirthDate'] = pd.to_datetime(BoardingDF['PassengerBirthDate'], format='%m/%d/%Y')
BoardingDF['FlightDate'] = pd.to_datetime(BoardingDF['FlightDate'], format='%Y-%m-%d')
BoardingDF['PassengerAge'] = ((BoardingDF['FlightDate'] - BoardingDF['PassengerBirthDate']).dt.days // 365.25).astype(int)

passport_series = BoardingDF['PassengerDocument'].astype(str).str.slice(0, 4)
BoardingDF['PassportSeriesLeft'] = passport_series.str.slice(0, 2)
BoardingDF['PassportSeriesRight'] = passport_series.str.slice(2, 4)

BoardingDF["BookingCode"] = (BoardingDF['BookingCode'] != 'Not presented').astype(int)

BoardingDF['Baggage'] = BoardingDF['Baggage'].fillna('No')

BoardingDF['BaggageTransit'] = (BoardingDF['Baggage'] == 'Transit').astype(int)
BoardingDF['BaggageDelayed'] = (BoardingDF['Baggage'] == 'Delayed').astype(int)
BoardingDF['BaggageRegistered'] = (BoardingDF['Baggage'] == 'Registered').astype(int)

BoardingDF['Flight_DayOfWeek'] = BoardingDF['FlightDate'].dt.dayofweek
BoardingDF['Flight_Month'] = BoardingDF['FlightDate'].dt.month

ft = BoardingDF['FlightTime'].astype(str)

BoardingDF['FlightTime'] = (
    ft.str.slice(0, 2).astype(int) * 60 +
    ft.str.slice(3, 5).astype(int)
)

BoardingDF["FlightNumber"] = BoardingDF['FlightNumber'].map(
    BoardingDF['FlightNumber'].value_counts(normalize=True)
)

BoardingDF["TicketNumber"] = (BoardingDF['TicketNumber'] != 'Not presented').astype(int)

BoardingDF["Destination"] = BoardingDF['Destination'].map(
    BoardingDF['Destination'].value_counts(normalize=True)
)

BoardingDF["CodeShare"] = (BoardingDF['CodeShare'] != 'Operated').astype(int)

BoardingDF = BoardingDF.drop(columns=['PassengerBirthDate', 'FlightDate', 'PassengerDocument', 'Baggage', 'FlightDate'])

In [5]:
display(BoardingDF)

,PassengerFirstName,PassengerSecondName,PassengerLastName,PassengerSex,BookingCode,TicketNumber,FlightTime,FlightNumber,CodeShare,Destination,PassengerAge,PassportSeriesLeft,PassportSeriesRight,BaggageTransit,BaggageDelayed,BaggageRegistered,Flight_DayOfWeek,Flight_Month
0,0.004654,0.002662,0.000786,1,1,1,365,0.004428,1,0.567346,34,28,79,1,0,0,2,3
1,0.004673,0.002424,0.000857,1,0,1,1330,0.001476,1,0.008914,41,17,88,0,0,1,5,3
2,0.004834,0.044899,0.001225,1,1,1,1330,0.001476,1,0.008914,26,43,96,1,0,0,5,3
3,0.005343,0.002494,0.000851,1,1,0,1330,0.001476,1,0.008914,19,25,95,0,0,1,5,3
4,0.004944,0.002598,0.000838,0,0,0,705,0.002849,1,0.567346,41,67,75,0,0,1,6,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155142,0.005027,0.045389,0.001141,1,1,1,390,0.001424,1,0.011273,23,32,56,0,1,0,3,5
155143,0.004918,0.002462,0.000709,1,0,1,390,0.001424,1,0.011273,42,34,97,0,0,0,3,5
155144,0.005292,0.002475,0.000928,0,0,0,450,0.001959,1,0.010848,44,66,18,0,0,1,0,10
155145,0.004589,0.009926,0.001044,1,0,1,420,0.001334,1,0.004460,17,45,67,0,0,1,5,9


# Нормализация FlightsDF

In [6]:
display(FlightsDF)

,Date,Codeshare,Flight,DepCity,DepAirport,DepCountry,ArrCity,ArrAirport,ArrCountry
0,2017-03-04,False,DL3377,Boston MA,BOS,United States of,Pittsburgh PA,PIT,United States of
1,2017-07-21,False,DL3630,Green Bay WI,GRB,United States of,Detroit MI,DTW,United States of
2,2017-03-22,False,DL5361,Pittsburgh PA,PIT,United States of,New York NY,LGA,United States of
3,2017-01-04,True,SU3702,Moscow,SVO,Russian Federation,Warsaw,WAW,Poland
4,2017-08-08,False,DL3412,Fayetteville AR,XNA,United States of,Atlanta GA,ATL,United States of
...,...,...,...,...,...,...,...,...,...
1308982,2017-12-15,False,DL3345,Washington,IAD,United States of,New York NY,JFK,United States of
1308983,2017-08-12,False,SU2014,Moscow,SVO,Russian Federation,Prague,PRG,Czech Republic
1308984,2017-12-29,False,DL2463,Atlanta GA,ATL,United States of,New York NY,LGA,United States of
1308985,2017-06-02,False,SU1463,Novosibirsk,OVB,Russian Federation,Moscow,SVO,Russian Federation


In [7]:
FlightsDF['Date'] = pd.to_datetime(FlightsDF['Date'])
FlightsDF['Flight_DayOfWeek'] = FlightsDF['Date'].dt.dayofweek
FlightsDF['Flight_Month'] = FlightsDF['Date'].dt.month

FlightsDF['Codeshare'] = FlightsDF['Codeshare'].astype(int)

FlightsDF["Flight"] = FlightsDF['Flight'].map(
    FlightsDF['Flight'].value_counts(normalize=True)
)

FlightsDF['Airport'] = FlightsDF['DepAirport'] + "_" + FlightsDF['ArrAirport']
FlightsDF["Airport"] = FlightsDF['Airport'].map(
    FlightsDF['Airport'].value_counts(normalize=True)
)

FlightsDF['City'] = FlightsDF['DepCity'] + "_" + FlightsDF['ArrCity']
FlightsDF["City"] = FlightsDF['City'].map(
    FlightsDF['City'].value_counts(normalize=True)
)

FlightsDF['Country'] = FlightsDF['DepCountry'] + "_" + FlightsDF['ArrCountry']
FlightsDF["Country"] = FlightsDF['Country'].map(
    FlightsDF['Country'].value_counts(normalize=True)
)

FlightsDF = FlightsDF.drop(columns=['Date', 'DepAirport', 'ArrAirport', 'DepCity', 'ArrCity', 'DepCountry', 'ArrCountry'])

In [8]:
display(FlightsDF)

,Codeshare,Flight,Flight_DayOfWeek,Flight_Month,Airport,City,Country
0,0,0.000083,5,3,0.000129,0.000129,0.129043
1,0,0.000058,4,7,0.000167,0.000167,0.129043
2,0,0.000032,2,3,0.000334,0.000474,0.129043
3,1,0.000186,2,1,0.000905,0.000905,0.000912
4,0,0.000062,1,8,0.000202,0.000202,0.129043
...,...,...,...,...,...,...,...
1308982,0,0.000080,4,12,0.000316,0.000723,0.129043
1308983,0,0.000168,5,8,0.003615,0.003615,0.005781
1308984,0,0.000024,4,12,0.000395,0.000827,0.129043
1308985,0,0.000342,4,6,0.001390,0.001390,0.119576


# Нормализация OperationalDF

In [9]:
display(OperationalDF)

,Date,Flight,From,To,Status,FF_Airline,FF_Number,Class,Fare
0,2017-01-01,AF1145,SVO,CDG,LANDED,FB,520518073,Y,YRSTFN
1,2017-01-01,AF1844,SVO,CDG,LANDED,FB,18427809,Y,YGRPZR
2,2017-01-01,AF4855,SVO,CDG,LANDED,KE,993927664,Y,YRSTJQ
3,2017-01-01,AF4901,SVO,CDG,LANDED,SU,789402704,Y,YGRPAT
4,2017-01-01,AZ545,SVO,FCO,LANDED,KE,352734424,Y,YFLXKO
...,...,...,...,...,...,...,...,...,...
1687048,2018-01-01,VN8067,SGN,VCS,LANDED,DT,182808803,Y,YGRPMT
1687049,2018-01-01,VN8067,SGN,VCS,LANDED,FB,88616357,Y,YFLXDJ
1687050,2018-01-01,VN813,SGN,REP,LANDED,DT,574004748,Y,YGRPAU
1687051,2018-01-01,VN8204,HAN,DIN,LANDED,KE,425304412,Y,YSTNDC


In [10]:
print(set(OperationalDF['Class']))

{'Y', 'A', 'J', 'P'}


In [11]:
OperationalDF['Date'] = pd.to_datetime(OperationalDF['Date'])
OperationalDF['Flight_DayOfWeek'] = OperationalDF['Date'].dt.dayofweek
OperationalDF['Flight_Month'] = OperationalDF['Date'].dt.month


OperationalDF = OperationalDF.drop(columns=['Date', 'Status'])

In [12]:
display(OperationalDF)

,Flight,From,To,FF_Airline,FF_Number,Class,Fare,Flight_DayOfWeek,Flight_Month
0,AF1145,SVO,CDG,FB,520518073,Y,YRSTFN,6,1
1,AF1844,SVO,CDG,FB,18427809,Y,YGRPZR,6,1
2,AF4855,SVO,CDG,KE,993927664,Y,YRSTJQ,6,1
3,AF4901,SVO,CDG,SU,789402704,Y,YGRPAT,6,1
4,AZ545,SVO,FCO,KE,352734424,Y,YFLXKO,6,1
...,...,...,...,...,...,...,...,...,...
1687048,VN8067,SGN,VCS,DT,182808803,Y,YGRPMT,0,1
1687049,VN8067,SGN,VCS,FB,88616357,Y,YFLXDJ,0,1
1687050,VN813,SGN,REP,DT,574004748,Y,YGRPAU,0,1
1687051,VN8204,HAN,DIN,KE,425304412,Y,YSTNDC,0,1
